In [1]:
from pathlib import Path
import altair as alt
import plotly.express as px
import polars as pl

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Polars:", pl.__version__)
print("Altair:", alt.__version__)

Polars: 2.1.0
Altair: 6.3.0


In [10]:
import polars as pl

raw_csv_path = RAW_DIR / "detroit_maternal.csv"
raw_csv_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_csv_path.exists():
  starter_csv = """indicator,year,detroit_pct,wayne_pct,michigan_pct,source_doc,notes
First-trimester prenatal care,2023,69.4,,,City of Detroit Maternal & Infant Health Data Brief,"Detroit births with first-trimester care (2023); Wayne & MI not included in this starter row"
Late or no prenatal care,2023,12.0,,,City of Detroit Maternal & Infant Health Data Brief,"Detroit births with late or no care (2023); Wayne & MI not included in this starter row"
WIC during pregnancy,2022,57.4,41.7,30.4,City of Detroit Maternal & Infant Health Data Brief,"Share of pregnant persons receiving WIC nutrition assistance"
Did not plan to breastfeed,2022,25.9,17.8,11.8,City of Detroit Maternal & Infant Health Data Brief,"Prenatal feeding plan; reflects support, access, and structural conditions"
Births under age 20,2019-2023,7.9,4.8,3.9,City of Detroit Maternal & Infant Health Data Brief,"Live births to birthing people under age 20"
Less than 12 years education,2022,20.6,14.0,9.6,City of Detroit Maternal & Infant Health Data Brief,"Educational attainment context among pregnant persons"
"""
  raw_csv_path.write_text(starter_csv, encoding="utf-8")

# 1. Read raw CSV as strings first so symbols/blanks don't break parsing
df_raw = pl.read_csv(raw_csv_path, infer_schema_length=0)

# 2. Normalize column names & clean text/numeric columns
text_cols = {"indicator", "year", "source_doc", "notes"}
pct_cols = ["detroit_pct", "wayne_pct", "michigan_pct"]

df_clean = (
    df_raw.rename(
        {c: c.strip().lower().replace(" ", "_") for c in df_raw.columns}
    )
    .with_columns(
        pl.col("indicator")
        .str.replace_all(r"[†*]+", "")
        .str.strip_chars()
        .alias("indicator"),
        pl.col("year").str.strip_chars().alias("year"),
        pl.col("source_doc").str.strip_chars().alias("source_doc"),
        pl.col("notes").str.strip_chars().alias("notes"),
    )
    .with_columns(
        pl.col(c)
        .str.replace_all(r"[%,\s]", "")
        .replace({
            "": None,
            "-": None,
            "—": None,
            "NA": None,
            "N/A": None,
            "*": None,
            ".": None,
        })
        .cast(pl.Float64, strict=False)
        .alias(c)
        for c in pct_cols
        if c in df_raw.columns
    )
    .with_columns(
        (pl.col("detroit_pct") - pl.col("michigan_pct"))
        .round(1)
        .alias("detroit_vs_michigan_pp")
    )
    .filter(pl.col("indicator").is_not_null() & (pl.col("indicator") != ""))
)

# 3. Save full cleaned table (all 6 rows + metadata)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_clean.write_parquet(PROCESSED_DIR / "detroit_maternal_clean.parquet")
df_clean.write_csv(PROCESSED_DIR / "detroit_maternal_clean.csv")

# 4. Filter to rows with complete 3-area comparisons for charting
df_complete_comparisons = df_clean.filter(
    pl.all_horizontal(pl.col(c).is_not_null() for c in pct_cols)
)

print("All cleaned rows:", df_clean.height)
print(df_clean)
print("\nComplete multi-area comparison rows:", df_complete_comparisons.height)
print(df_complete_comparisons.select(["indicator", "year", *pct_cols, "detroit_vs_michigan_pp"]))

All cleaned rows: 6
shape: (6, 8)
┌────────────┬───────────┬────────────┬───────────┬────────────┬───────────┬───────────┬───────────┐
│ indicator  ┆ year      ┆ detroit_pc ┆ wayne_pct ┆ michigan_p ┆ source_do ┆ notes     ┆ detroit_v │
│ ---        ┆ ---       ┆ t          ┆ ---       ┆ ct         ┆ c         ┆ ---       ┆ s_michiga │
│ str        ┆ str       ┆ ---        ┆ f64       ┆ ---        ┆ ---       ┆ str       ┆ n_pp      │
│            ┆           ┆ f64        ┆           ┆ f64        ┆ str       ┆           ┆ ---       │
│            ┆           ┆            ┆           ┆            ┆           ┆           ┆ f64       │
╞════════════╪═══════════╪════════════╪═══════════╪════════════╪═══════════╪═══════════╪═══════════╡
│ First-trim ┆ 2023      ┆ 69.4       ┆ null      ┆ null       ┆ City of   ┆ Detroit   ┆ null      │
│ ester      ┆           ┆            ┆           ┆            ┆ Detroit   ┆ births    ┆           │
│ prenatal   ┆           ┆            ┆           ┆      

In [12]:
df_long = (
    df_complete_comparisons.unpivot(
        index=["indicator", "year", "source_doc", "notes", "detroit_vs_michigan_pp"],
        on=["detroit_pct", "wayne_pct", "michigan_pct"],
        variable_name="geo_key",
        value_name="pct",
    )
    .with_columns(
        pl.col("geo_key")
        .replace({
            "detroit_pct": "Detroit",
            "wayne_pct": "Wayne County",
            "michigan_pct": "Michigan",
        })
        .alias("geography")
    )
    .drop("geo_key")
)

chart = (
    alt.Chart(df_long)
    .mark_bar()
    .encode(
        x=alt.X("pct:Q", title="Share (%)"),
        y=alt.Y(
            "geography:N",
            sort=["Detroit", "Wayne County", "Michigan"],
            title=None,
        ),
        color=alt.Color(
            "geography:N",
            sort=["Detroit", "Wayne County", "Michigan"],
            title="Geography",
        ),
        row=alt.Row(
            "indicator:N",
            header=alt.Header(title=None, labelAngle=0, labelAlign="left"),
        ),
        tooltip=["indicator", "year", "geography", "pct", "detroit_vs_michigan_pp", "notes"],
    )
    .properties(
        width=420,
        height=60,
        title="Detroit Maternal & Structural Health Indicators vs. Wayne County & Michigan",
    )
)

chart

alt.Chart(...)

In [13]:
df_disparity = (
    df_clean.filter(
        pl.all_horizontal(
            pl.col(c).is_not_null()
            for c in ["detroit_pct", "wayne_pct", "michigan_pct"]
        )
    )
    .with_columns(
        (pl.col("detroit_pct") - pl.col("michigan_pct"))
        .round(1)
        .alias("detroit_vs_michigan_pp"),
        (pl.col("detroit_pct") - pl.col("wayne_pct"))
        .round(1)
        .alias("detroit_vs_wayne_pp"),
        (pl.col("detroit_pct") / pl.col("michigan_pct"))
        .round(2)
        .alias("detroit_to_michigan_ratio"),
    )
    .sort("detroit_to_michigan_ratio", descending=True)
)

print(
    df_disparity.select([
        "indicator",
        "year",
        "detroit_pct",
        "wayne_pct",
        "michigan_pct",
        "detroit_vs_michigan_pp",
        "detroit_to_michigan_ratio",
    ])
)

shape: (4, 7)
┌───────────────┬───────────┬─────────────┬───────────┬──────────────┬──────────────┬──────────────┐
│ indicator     ┆ year      ┆ detroit_pct ┆ wayne_pct ┆ michigan_pct ┆ detroit_vs_m ┆ detroit_to_m │
│ ---           ┆ ---       ┆ ---         ┆ ---       ┆ ---          ┆ ichigan_pp   ┆ ichigan_rati │
│ str           ┆ str       ┆ f64         ┆ f64       ┆ f64          ┆ ---          ┆ o            │
│               ┆           ┆             ┆           ┆              ┆ f64          ┆ ---          │
│               ┆           ┆             ┆           ┆              ┆              ┆ f64          │
╞═══════════════╪═══════════╪═════════════╪═══════════╪══════════════╪══════════════╪══════════════╡
│ Did not plan  ┆ 2022      ┆ 25.9        ┆ 17.8      ┆ 11.8         ┆ 14.1         ┆ 2.19         │
│ to breastfeed ┆           ┆             ┆           ┆              ┆              ┆              │
│ Less than 12  ┆ 2022      ┆ 20.6        ┆ 14.0      ┆ 9.6          ┆ 11.0  

In [14]:
indicator_order = df_disparity["indicator"].to_list()

df_overview_long = (
    df_disparity.unpivot(
        index=[
            "indicator",
            "year",
            "detroit_vs_michigan_pp",
            "detroit_to_michigan_ratio",
            "notes",
        ],
        on=["detroit_pct", "wayne_pct", "michigan_pct"],
        variable_name="geo_col",
        value_name="pct",
    )
    .with_columns(
        pl.col("geo_col")
        .replace({
            "detroit_pct": "Detroit",
            "wayne_pct": "Wayne County",
            "michigan_pct": "Michigan",
        })
        .alias("geography")
    )
    .drop("geo_col")
)

overview_chart = (
    alt.Chart(df_overview_long)
    .mark_bar()
    .encode(
        y=alt.Y(
            "indicator:N",
            sort=indicator_order,
            title=None,
        ),
        x=alt.X("pct:Q", title="Share (%)"),
        yOffset=alt.YOffset(
            "geography:N",
            sort=["Detroit", "Wayne County", "Michigan"],
        ),
        color=alt.Color(
            "geography:N",
            sort=["Detroit", "Wayne County", "Michigan"],
            scale=alt.Scale(
                domain=["Detroit", "Wayne County", "Michigan"],
                range=["#6b2d5c", "#9aa5b1", "#cbd2d9"],
            ),
            title="Geography",
        ),
        tooltip=[
            "indicator",
            "year",
            "geography",
            "pct",
            "detroit_vs_michigan_pp",
            "detroit_to_michigan_ratio",
            "notes",
        ],
    )
    .properties(
        width=480,
        height=260,
        title="All Multi-Area Indicators Ranked by Detroit-to-Michigan Ratio",
    )
)

overview_chart

alt.Chart(...)